# NeRF: Neural Radiance Fields for View Synthesis

**A Comprehensive PyTorch Implementation from Scratch**

Reference: Tancik et al., "NeRF: Representing Scenes as Neural Radiance Fields for View Synthesis" (2020)
https://www.matthewtancik.com/nerf

This notebook implements the complete NeRF pipeline:
- **Ray generation**: Camera intrinsics/extrinsics → rays in 3D space
- **Positional encoding**: Fourier features for high-frequency learning
- **Neural radiance field**: MLP mapping (position, direction) → (RGB, density)
- **Volumetric rendering**: Alpha compositing along rays
- **Hierarchical sampling**: Importance weighting for efficient rendering
- **Training**: Photometric MSE loss on multi-view images
- **Novel view synthesis**: Render new camera viewpoints

This is a classical **neural ray tracing** pipeline where explicit geometry is replaced by a learned implicit representation.

## 1. Import Required Libraries

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch
from mpl_toolkits.mplot3d import proj3d
import seaborn as sns

# Configure for better visualization
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 8)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
print(f"PyTorch version: {torch.__version__}")
print(f"NumPy version: {np.__version__}")

## 2. Positional Encoding (Fourier Features)

**Key Insight**: Neural networks struggle to learn high-frequency variations in image space. The positional encoding maps continuous input coordinates to high-dimensional Fourier features:

$$\gamma(p) = \left(\sin(2^0 \pi p), \cos(2^0 \pi p), \ldots, \sin(2^{L-1} \pi p), \cos(2^{L-1} \pi p)\right)$$

Where:
- $p$ is the input coordinate (position or direction)
- $L$ is the number of frequency levels
- Output dimension: $2L \times \text{input\_dim}$

This allows the network to fit high-frequency details without requiring exponentially more neurons.

In [ ]:
class PositionalEncoding(nn.Module):
    """
    Fourier positional encoding for continuous input coordinates.
    
    Transforms scalars or vectors to high-dimensional Fourier features.
    """
    
    def __init__(self, num_freqs=10, include_input=True, log_space=True):
        super().__init__()
        self.num_freqs = num_freqs
        self.include_input = include_input
        self.log_space = log_space
        
        # Pre-compute frequency bands: 2^0, 2^1, ..., 2^(L-1)
        if log_space:
            freq_bands = torch.linspace(0.0, num_freqs - 1, num_freqs)
            self.register_buffer('freq_bands', 2.0 ** freq_bands * np.pi)
        else:
            freq_bands = torch.linspace(0.0, num_freqs - 1, num_freqs)
            self.register_buffer('freq_bands', freq_bands * np.pi)
    
    def forward(self, x):
        """
        Args:
            x (torch.Tensor): Input of shape (..., D)
        
        Returns:
            torch.Tensor: Encoded output of shape (..., D + 2*D*L) if include_input
        """
        # Expand: (..., D) -> (..., D, 1)
        x_expanded = x.unsqueeze(-1)
        
        # Scale by frequencies: (..., D, 1) * (L,) -> (..., D, L)
        scaled = x_expanded * self.freq_bands
        
        # Compute sin and cos
        sin_features = torch.sin(scaled)  # (..., D, L)
        cos_features = torch.cos(scaled)  # (..., D, L)
        
        # Interleave: (..., D, 2*L)
        encoded = torch.cat([sin_features, cos_features], dim=-1)
        
        # Flatten last two dimensions: (..., D*2*L)
        *batch_shape, d, freq_dim = encoded.shape
        encoded = encoded.view(*batch_shape, d * freq_dim)
        
        if self.include_input:
            return torch.cat([x, encoded], dim=-1)
        else:
            return encoded
    
    def get_output_dim(self, input_dim):
        """Compute output dimension for given input dimension."""
        encoded_dim = 2 * input_dim * self.num_freqs
        return (input_dim + encoded_dim) if self.include_input else encoded_dim


# Demonstrate positional encoding
print("Positional Encoding Demonstration")
print("=" * 50)

encoder = PositionalEncoding(num_freqs=4, include_input=True)
test_input = torch.tensor([[[0.5]]], dtype=torch.float32)  # Shape: [1, 1, 1]
encoded = encoder(test_input)

print(f"Input: {test_input.shape} -> {test_input.squeeze().item():.3f}")
print(f"Encoded: {encoded.shape}")
print(f"Output dimension: {encoder.get_output_dim(1)} (input + 2*L*input_dim = 1 + 2*4*1)")
print(f"\nFirst 4 encoded values: {encoded.squeeze()[:4].detach().numpy()}")

# Visualize encoding
x_vals = np.linspace(-np.pi, np.pi, 100)
encoder_viz = PositionalEncoding(num_freqs=5, include_input=False)
x_tensor = torch.tensor(x_vals, dtype=torch.float32).unsqueeze(-1)
encoded_vals = encoder_viz(x_tensor).detach().numpy()

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i in range(min(6, encoded_vals.shape[1])):
    axes[i].plot(x_vals, encoded_vals[:, i], linewidth=2)
    axes[i].set_title(f'Encoded Dimension {i+1}')
    axes[i].grid(True, alpha=0.3)
    axes[i].set_xlabel('Input Value')

plt.tight_layout()
plt.suptitle('Fourier Features from Positional Encoding', y=1.02, fontsize=14)
plt.show()

## 3. Neural Radiance Field Network Architecture

The core NeRF network is an MLP that maps encoded 3D coordinates and viewing direction to:
- **Density (σ)**: View-independent volume density (opacity)
- **RGB color**: View-dependent color

**Architecture**:
- Input: Positional encoding of (x, y, z) + positional encoding of view direction
- 8 hidden layers with 256 units each
- ReLU activations, skip connection at layer 4
- Output branches:
  - **Density branch**: Linear layer → softplus → σ ∈ [0, ∞)
  - **Color branch**: ReLU → Linear → sigmoid → RGB ∈ [0, 1]

This design ensures:
- Density is **view-independent** (captures geometry)
- Color is **view-dependent** (captures appearance/reflectance)

In [ ]:
class NeRFNetwork(nn.Module):
    """
    Multi-layer perceptron for neural radiance field.
    
    Maps (encoded position, encoded direction) -> (RGB, density)
    """
    
    def __init__(
        self,
        input_dim_pos,
        input_dim_dir=24,
        hidden_dim=256,
        num_layers=8,
        skip_layers=None,
    ):
        super().__init__()
        
        self.input_dim_pos = input_dim_pos
        self.input_dim_dir = input_dim_dir
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        self.skip_layers = skip_layers or [4]
        
        # Position processing stream
        self.fc_pos_input = nn.Linear(input_dim_pos, hidden_dim)
        
        self.pos_layers = nn.ModuleList()
        for i in range(1, num_layers):
            layer_input_dim = hidden_dim
            if i in self.skip_layers:
                layer_input_dim += input_dim_pos
            self.pos_layers.append(nn.Linear(layer_input_dim, hidden_dim))
        
        # View-independent density output
        self.fc_density = nn.Linear(hidden_dim, 1)
        
        # View-dependent RGB branch
        self.fc_rgb_input = nn.Linear(hidden_dim + input_dim_dir, hidden_dim)
        self.fc_rgb_output = nn.Linear(hidden_dim, 3)
        
        self.relu = nn.ReLU()
        self.sigmoid = nn.Sigmoid()
        self.softplus = nn.Softplus()
    
    def forward(self, pos_encoded, dir_encoded):
        """
        Args:
            pos_encoded: Encoded position [batch, input_dim_pos]
            dir_encoded: Encoded direction [batch, input_dim_dir]
        
        Returns:
            rgb: [batch, 3] in [0, 1]
            density: [batch, 1] in [0, ∞)
        """
        x_pos = pos_encoded
        
        # Process through position stream with skip connections
        x = self.relu(self.fc_pos_input(x_pos))
        
        for i, layer in enumerate(self.pos_layers):
            if i + 1 in self.skip_layers:
                x = torch.cat([x, x_pos], dim=-1)
            x = self.relu(layer(x))
        
        # Density (view-independent)
        density = self.softplus(self.fc_density(x))
        
        # RGB (view-dependent)
        x_rgb = torch.cat([x, dir_encoded], dim=-1)
        x_rgb = self.relu(self.fc_rgb_input(x_rgb))
        rgb = self.sigmoid(self.fc_rgb_output(x_rgb))
        
        return rgb, density


# Test network
print("NeRF Network Test")
print("=" * 50)

num_freqs_pos = 10
num_freqs_dir = 4

pos_encoder = PositionalEncoding(num_freqs=num_freqs_pos, include_input=True)
dir_encoder = PositionalEncoding(num_freqs=num_freqs_dir, include_input=True)

pos_dim = pos_encoder.get_output_dim(3)
dir_dim = dir_encoder.get_output_dim(3)

print(f"Position encoding dim: {pos_dim} (3 coords + 2*10*3 Fourier)")
print(f"Direction encoding dim: {dir_dim} (3 direction + 2*4*3 Fourier)")

network = NeRFNetwork(pos_dim, dir_dim).to(device)

# Test forward pass
batch_size = 10
pos_sample = torch.randn(batch_size, 3, device=device)
dir_sample = torch.randn(batch_size, 3, device=device)
dir_sample = dir_sample / torch.norm(dir_sample, dim=-1, keepdim=True)

pos_encoded = pos_encoder(pos_sample)
dir_encoded = dir_encoder(dir_sample)

rgb, density = network(pos_encoded, dir_encoded)

print(f"\nNetwork output shapes:")
print(f"  RGB: {rgb.shape} (values in [0, 1]: [{rgb.min():.4f}, {rgb.max():.4f}])")
print(f"  Density: {density.shape} (values in [0, ∞): [{density.min():.4f}, {density.max():.4f}])")

# Count parameters
num_params = sum(p.numel() for p in network.parameters())
print(f"\nTotal parameters: {num_params:,}")

## 4. Ray Generation from Camera Parameters

**Classical Ray Tracing Pipeline**:

Given a camera with:
- **Intrinsics**: Focal length $f$ (field of view)
- **Extrinsics**: Pose matrix $[\mathbf{R} | \mathbf{t}]$ (rotation + translation)

For each pixel $(u, v)$:
1. Normalize to camera space: $x_c = (u - c_x) / f$, $y_c = (v - c_y) / f$, $z_c = 1$
2. Transform to world space: $\mathbf{r} = \mathbf{R}^T [\mathbf{x}_c, 1]$
3. Ray: $\mathbf{R}(t) = \mathbf{O} + t \mathbf{D}$ where $\mathbf{O}$ is camera center, $\mathbf{D}$ is direction

In [ ]:
def get_rays(height, width, focal, pose):
    """
    Generate rays from camera intrinsics and extrinsics.
    
    Args:
        height (int): Image height
        width (int): Image width
        focal (float): Focal length
        pose (torch.Tensor): Camera pose [4, 4]
    
    Returns:
        rays_o: Ray origins [height, width, 3]
        rays_d: Ray directions [height, width, 3]
    """
    # Create pixel coordinates
    x = torch.arange(width, dtype=torch.float32)
    y = torch.arange(height, dtype=torch.float32)
    xx, yy = torch.meshgrid(x, y, indexing='ij')
    
    # Normalize to camera space
    coords = torch.stack([
        (xx - width / 2.0) / focal,
        -(yy - height / 2.0) / focal,
        torch.ones_like(xx),
    ], dim=-1)  # [width, height, 3]
    
    # Extract rotation and translation
    R = pose[:3, :3]
    t = pose[:3, 3]
    R_inv = R.T
    
    # Transform directions to world space
    dirs = torch.matmul(coords, R_inv.T)
    dirs = dirs / (torch.norm(dirs, dim=-1, keepdim=True) + 1e-8)
    
    # Ray origin (camera center in world space)
    rays_o = -torch.matmul(R_inv, t.unsqueeze(-1)).squeeze(-1)
    rays_o = rays_o.expand(*coords.shape[:-1], 3)
    
    return rays_o.transpose(0, 1), dirs.transpose(0, 1)


# Test ray generation
print("Ray Generation Test")
print("=" * 50)

height, width = 64, 64
focal = 50.0

# Simple pose: identity
pose_identity = torch.eye(4)
rays_o, rays_d = get_rays(height, width, focal, pose_identity)

print(f"Image size: {height}x{width}, focal length: {focal}")
print(f"Ray origins shape: {rays_o.shape}")
print(f"Ray directions shape: {rays_d.shape}")
print(f"\nRay origin at center: {rays_o[height//2, width//2].numpy()}")
print(f"Ray direction at center: {rays_d[height//2, width//2].numpy()}")

# Visualize rays (subset)
fig = plt.figure(figsize=(12, 5))

# Ray directions visualization
ax1 = fig.add_subplot(121)
# Sample a grid of directions
step = 8
y_indices = torch.arange(0, height, step)
x_indices = torch.arange(0, width, step)
for y in y_indices:
    for x in x_indices:
        d = rays_d[y, x].numpy()
        ax1.arrow(x, y, d[0]*5, -d[1]*5, head_width=1, head_length=0.5, alpha=0.5)
ax1.set_xlim(-5, width+5)
ax1.set_ylim(-5, height+5)
ax1.set_aspect('equal')
ax1.set_title('Ray Directions (projected to 2D)')
ax1.invert_yaxis()
ax1.grid(True, alpha=0.3)

# 3D ray visualization
ax2 = fig.add_subplot(122, projection='3d')
# Sample fewer rays for clarity
step = 16
for y in torch.arange(0, height, step):
    for x in torch.arange(0, width, step):
        o = rays_o[y, x].numpy()
        d = rays_d[y, x].numpy()
        ax2.quiver(o[0], o[1], o[2], d[0], d[1], d[2], 
                   length=0.5, alpha=0.6, arrow_length_ratio=0.3)

ax2.set_xlabel('X')
ax2.set_ylabel('Y')
ax2.set_zlabel('Z')
ax2.set_title('3D Ray Origins and Directions')
ax2.set_xlim(-1, 1)
ax2.set_ylim(-1, 1)
ax2.set_zlim(-1, 1)

plt.tight_layout()
plt.show()

## 5. Stratified and Hierarchical Sampling

**Stratified Sampling**: Divide the interval $[t_n, t_f]$ into $N$ bins and sample uniformly within each bin:

$$t_i = t_n + \frac{i + \epsilon}{N}(t_f - t_n), \quad \epsilon \sim \text{Uniform}(0, 1)$$

**Hierarchical Importance Sampling**: After coarse rendering, use density to weight sampling:
1. Compute PDF from coarse network density
2. Sample inversely from CDF to concentrate samples in high-density regions
3. Combine with coarse samples and re-render with fine network

In [ ]:
def stratified_sample(rays_o, rays_d, near, far, num_samples, perturb=True):
    """
    Stratified sampling along rays.
    
    Args:
        rays_o: Ray origins [batch, 3]
        rays_d: Ray directions [batch, 3]
        near (float): Near plane
        far (float): Far plane
        num_samples (int): Number of samples per ray
        perturb (bool): Add random perturbation
    
    Returns:
        sample_points: 3D coordinates [batch, num_samples, 3]
        depths: Depth values [batch, num_samples]
    """
    batch_size = rays_o.shape[0]
    
    # Linear depth sampling
    depths = torch.linspace(near, far, num_samples, device=rays_o.device)
    depths = depths.unsqueeze(0).expand(batch_size, -1)
    
    if perturb:
        # Add random perturbation within bins
        bin_width = (far - near) / num_samples
        depths = depths + (torch.rand_like(depths) - 0.5) * bin_width
        depths = torch.clamp(depths, near, far)
    
    # Compute 3D points: P = O + d * D
    sample_points = rays_o.unsqueeze(1) + rays_d.unsqueeze(1) * depths.unsqueeze(-1)
    
    return sample_points, depths


def hierarchical_sample(rays_o, rays_d, depths_coarse, weights, num_samples, perturb=True):
    """
    Importance sampling based on coarse network weights.
    
    Args:
        rays_o: Ray origins [batch, 3]
        rays_d: Ray directions [batch, 3]
        depths_coarse: Coarse depths [batch, num_coarse]
        weights: Density weights [batch, num_coarse]
        num_samples (int): Number of new samples
        perturb (bool): Add perturbation
    
    Returns:
        sample_points: New 3D coordinates [batch, num_samples, 3]
        depths_new: New depth values [batch, num_samples]
        depths_all: All depths combined and sorted [batch, num_coarse + num_samples]
    """
    batch_size = rays_o.shape[0]
    
    # Normalize weights to PDF
    pdf = weights / (torch.sum(weights, dim=-1, keepdim=True) + 1e-5)
    cdf = torch.cumsum(pdf, dim=-1)
    cdf = torch.cat([torch.zeros_like(cdf[..., :1]), cdf], dim=-1)
    
    # Inverse transform sampling
    uniform_samples = torch.rand(batch_size, num_samples, device=rays_o.device)
    indices = torch.searchsorted(cdf, uniform_samples, right=False) - 1
    indices = torch.clamp(indices, 0, cdf.shape[-1] - 2)
    
    # Linear interpolation
    depths_sorted = torch.sort(depths_coarse, dim=-1)[0]
    left_edges = depths_sorted[torch.arange(batch_size).unsqueeze(-1), indices]
    right_edges = depths_sorted[torch.arange(batch_size).unsqueeze(-1), indices + 1]
    depths_new = left_edges + torch.rand_like(left_edges) * (right_edges - left_edges)
    
    # Compute sample points
    sample_points = rays_o.unsqueeze(1) + rays_d.unsqueeze(1) * depths_new.unsqueeze(-1)
    
    # Combine and sort
    depths_all = torch.cat([depths_coarse, depths_new], dim=-1)
    depths_all = torch.sort(depths_all, dim=-1)[0]
    
    return sample_points, depths_new, depths_all


# Test sampling
print("Stratified and Hierarchical Sampling Test")
print("=" * 50)

batch_size = 4
rays_o_test = torch.randn(batch_size, 3)
rays_d_test = torch.randn(batch_size, 3)
rays_d_test = rays_d_test / torch.norm(rays_d_test, dim=-1, keepdim=True)

# Stratified sampling
near, far = 2.0, 6.0
num_coarse = 32
sample_points, depths = stratified_sample(rays_o_test, rays_d_test, near, far, num_coarse)

print(f"Stratified sampling:")
print(f"  Sample points: {sample_points.shape}")
print(f"  Depths: {depths.shape}")
print(f"  Depth range: [{depths.min():.3f}, {depths.max():.3f}]")

# Visualize sampling distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Stratified sampling
axes[0].scatter(np.arange(num_coarse), depths[0].numpy(), alpha=0.6, s=50)
axes[0].set_xlabel('Sample Index')
axes[0].set_ylabel('Depth')
axes[0].set_title('Stratified Sampling: Uniform Depths')
axes[0].grid(True, alpha=0.3)
axes[0].set_ylim(near - 0.5, far + 0.5)

# Hierarchical sampling
weights = torch.exp(-((depths[0] - 4.0) ** 2) / 0.5)  # Simulate Gaussian density
num_fine = 64
sample_points_fine, depths_fine, depths_all = hierarchical_sample(
    rays_o_test, rays_d_test, depths, weights.unsqueeze(0), num_fine
)

# Plot 2: Hierarchical sampling
axes[1].scatter(np.arange(len(depths[0])), depths[0].numpy(), label='Coarse', alpha=0.6, s=50)
axes[1].scatter(np.arange(len(depths_fine[0])) + 0.2, depths_fine[0].numpy(), 
               label='Fine', alpha=0.6, s=30, color='orange')
axes[1].set_xlabel('Sample Index')
axes[1].set_ylabel('Depth')
axes[1].set_title('Hierarchical Sampling: Importance-Weighted')
axes[1].legend()
axes[1].grid(True, alpha=0.3)
axes[1].set_ylim(near - 0.5, far + 0.5)

plt.tight_layout()
plt.show()

## 6. Volumetric Rendering with Alpha Compositing

**Volume Rendering Equation**:
$$C(\mathbf{r}) = \int_{t_n}^{t_f} T(t) \sigma(\mathbf{r}(t)) \mathbf{c}(\mathbf{r}(t), \mathbf{d}) dt$$

Where:
- $\sigma(\mathbf{r}(t))$ is density (opacity) at point $\mathbf{r}(t)$
- $\mathbf{c}(\mathbf{r}(t), \mathbf{d})$ is color
- $T(t) = \exp\left(-\int_{t_n}^t \sigma(\mathbf{r}(s)) ds\right)$ is transmittance

**Discrete Approximation (Alpha Compositing)**:
$$\alpha_i = 1 - \exp(-\sigma_i \delta_i)$$
$$T_i = \exp\left(-\sum_{j=1}^{i-1} \sigma_j \delta_j\right)$$
$$C = \sum_i T_i \alpha_i \mathbf{c}_i + T_N \mathbf{c}_{bg}$$

This is fully differentiable and enables end-to-end training!

In [ ]:
def volume_rendering(rgb, density, depths, rays_d, bg_color=None, white_background=False):
    """
    Differentiable volume rendering via alpha compositing.
    
    Args:
        rgb: Colors [batch, num_samples, 3]
        density: Density σ [batch, num_samples, 1]
        depths: Depth values [batch, num_samples]
        rays_d: Ray directions [batch, 3]
        bg_color: Background color [3] or [batch, 3]
        white_background (bool): Use white background
    
    Returns:
        dict with 'color', 'depth', 'acc', 'weights', 'transmittance'
    """
    batch_size = density.shape[0]
    
    # Compute step sizes
    depth_diff = torch.diff(depths, dim=-1, prepend=depths[..., :1] - 1e6)
    depth_diff = torch.abs(depth_diff)
    depth_diff = torch.clamp(depth_diff, min=1e-3)
    
    # Compute alpha: α_i = 1 - exp(-σ_i * δ_i)
    alpha = 1.0 - torch.exp(-density.squeeze(-1) * depth_diff)
    alpha = torch.clamp(alpha, 0.0, 1.0)
    
    # Compute transmittance
    dists = density.squeeze(-1) * depth_diff
    transmittance = torch.exp(-torch.cumsum(dists, dim=-1) + dists)
    transmittance = torch.cat([torch.ones_like(transmittance[..., :1]), transmittance[..., :-1]], dim=-1)
    
    # Composite weights: w_i = T_i * α_i
    weights = transmittance * alpha
    
    # Composite color: C = Σ w_i * c_i
    color = torch.sum(weights.unsqueeze(-1) * rgb, dim=1)
    
    # Expected depth
    depth_map = torch.sum(weights * depths, dim=-1, keepdim=True)
    
    # Accumulated alpha
    acc = torch.sum(weights, dim=-1, keepdim=True)
    
    # Add background
    if bg_color is None:
        bg_color = torch.ones(3, device=rgb.device) if white_background else torch.zeros(3, device=rgb.device)
    
    if bg_color.dim() == 1:
        bg_color = bg_color.unsqueeze(0).expand(batch_size, -1)
    
    color = color + (1.0 - acc) * bg_color
    
    return {
        'color': color,
        'depth': depth_map,
        'acc': acc,
        'weights': weights,
        'transmittance': transmittance,
    }


# Test volume rendering
print("Volume Rendering Test")
print("=" * 50)

batch_size = 2
num_samples = 32

# Synthetic density (Gaussian blob)
depths_test = torch.linspace(2.0, 6.0, num_samples)
density_test = torch.exp(-((depths_test - 4.0) ** 2) / 0.5).unsqueeze(0).unsqueeze(-1)
density_test = density_test.expand(batch_size, -1, 1)

# Random colors
rgb_test = torch.rand(batch_size, num_samples, 3)

# Random rays
rays_d_test = torch.randn(batch_size, 3)
rays_d_test = rays_d_test / torch.norm(rays_d_test, dim=-1, keepdim=True)

# Render
render_dict = volume_rendering(rgb_test, density_test, depths_test, rays_d_test)

print(f"Rendered color: {render_dict['color'].shape}")
print(f"  Value range: [{render_dict['color'].min():.3f}, {render_dict['color'].max():.3f}]")
print(f"Rendered depth: {render_dict['depth'].shape}")
print(f"  Value range: [{render_dict['depth'].min():.3f}, {render_dict['depth'].max():.3f}]")
print(f"Accumulated alpha: {render_dict['acc'].shape}")
print(f"  Value range: [{render_dict['acc'].min():.3f}, {render_dict['acc'].max():.3f}]")

# Visualize rendering components
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

# Density
axes[0, 0].plot(depths_test.numpy(), density_test[0, :, 0].numpy(), linewidth=2)
axes[0, 0].fill_between(depths_test.numpy(), density_test[0, :, 0].numpy(), alpha=0.3)
axes[0, 0].set_title('Density σ(t)')
axes[0, 0].set_xlabel('Depth')
axes[0, 0].grid(True, alpha=0.3)

# Alpha
alpha_vals = 1.0 - torch.exp(-density_test[0, :, 0] * 0.1)
axes[0, 1].bar(np.arange(num_samples), alpha_vals.numpy(), alpha=0.7)
axes[0, 1].set_title('Alpha α_i = 1 - exp(-σ_i δ_i)')
axes[0, 1].set_xlabel('Sample Index')
axes[0, 1].grid(True, alpha=0.3, axis='y')

# Transmittance
transmittance_vals = render_dict['transmittance'][0].numpy()
axes[0, 2].plot(np.arange(num_samples), transmittance_vals, linewidth=2, marker='o')
axes[0, 2].set_title('Transmittance T_i')
axes[0, 2].set_xlabel('Sample Index')
axes[0, 2].grid(True, alpha=0.3)

# Weights
weights_vals = render_dict['weights'][0].numpy()
axes[1, 0].bar(np.arange(num_samples), weights_vals, alpha=0.7, color='green')
axes[1, 0].set_title('Composite Weights w_i = T_i α_i')
axes[1, 0].set_xlabel('Sample Index')
axes[1, 0].grid(True, alpha=0.3, axis='y')

# Color channel
colors_vals = render_dict['color'][0].numpy()
axes[1, 1].bar(['R', 'G', 'B'], colors_vals, color=['red', 'green', 'blue'], alpha=0.7)
axes[1, 1].set_title('Composite Color')
axes[1, 1].set_ylim(0, 1)
axes[1, 1].grid(True, alpha=0.3, axis='y')

# Rendered image (visualize as color)
img = render_dict['color'][0].unsqueeze(0).unsqueeze(0).numpy()
axes[1, 2].imshow(img)
axes[1, 2].set_title('Rendered Pixel')
axes[1, 2].axis('off')

plt.tight_layout()
plt.show()

## 7. Training Loop with Photometric Loss

**Loss Function**:
$$\mathcal{L} = \text{MSE}(C_{\text{coarse}}, C_{gt}) + \text{MSE}(C_{\text{fine}}, C_{gt})$$

Where $C_{gt}$ is the ground truth image and $C_{\text{coarse}}, C_{\text{fine}}$ are rendered colors from coarse and fine networks.

**Training Algorithm**:
1. Sample rays and pixels from training images
2. Forward through coarse network → render coarse image
3. Forward through fine network (with importance sampling) → render fine image
4. Compute MSE loss on both
5. Backpropagate and update network weights
6. Repeat with new rays

In [ ]:
class SimpleNeRFTrainer:
    """Simplified NeRF trainer for demonstration."""
    
    def __init__(self, coarse_net, fine_net, pos_encoder, dir_encoder, lr=5e-4):
        self.coarse = coarse_net
        self.fine = fine_net
        self.pos_encoder = pos_encoder
        self.dir_encoder = dir_encoder
        self.optimizer = torch.optim.Adam(
            list(coarse_net.parameters()) + list(fine_net.parameters()),
            lr=lr
        )
        self.criterion = nn.MSELoss()
        self.losses = []
    
    def render_rays(self, rays_o, rays_d, near=2.0, far=6.0, 
                   num_coarse=32, num_fine=64, white_bg=False):
        """Complete rendering pipeline."""
        batch_size = rays_o.shape[0]
        
        # Stratified sampling
        sample_points_coarse, depths_coarse = stratified_sample(
            rays_o, rays_d, near, far, num_coarse, perturb=True
        )
        
        # Encode positions and directions
        pos_enc = self.pos_encoder(sample_points_coarse.reshape(-1, 3))
        pos_enc = pos_enc.reshape(batch_size, num_coarse, -1)
        
        dirs_broadcast = rays_d.unsqueeze(1).expand(batch_size, num_coarse, 3).reshape(-1, 3)
        dir_enc = self.dir_encoder(dirs_broadcast)
        dir_enc = dir_enc.reshape(batch_size, num_coarse, -1)
        
        # Coarse network
        rgb_coarse, density_coarse = self.coarse(
            pos_enc.reshape(-1, pos_enc.shape[-1]),
            dir_enc.reshape(-1, dir_enc.shape[-1])
        )
        rgb_coarse = rgb_coarse.reshape(batch_size, num_coarse, 3)
        density_coarse = density_coarse.reshape(batch_size, num_coarse, 1)
        
        # Render coarse
        render_coarse = volume_rendering(rgb_coarse, density_coarse, depths_coarse, rays_d, white_bg)
        
        # Hierarchical sampling
        weights_coarse = render_coarse['weights']
        sample_points_fine, depths_fine, depths_all = hierarchical_sample(
            rays_o, rays_d, depths_coarse, weights_coarse, num_fine
        )
        
        # Encode fine samples
        pos_enc_fine = self.pos_encoder(sample_points_fine.reshape(-1, 3))
        pos_enc_fine = pos_enc_fine.reshape(batch_size, num_fine, -1)
        
        dirs_broadcast_fine = rays_d.unsqueeze(1).expand(batch_size, num_fine, 3).reshape(-1, 3)
        dir_enc_fine = self.dir_encoder(dirs_broadcast_fine)
        dir_enc_fine = dir_enc_fine.reshape(batch_size, num_fine, -1)
        
        # Fine network
        rgb_fine, density_fine = self.fine(
            pos_enc_fine.reshape(-1, pos_enc_fine.shape[-1]),
            dir_enc_fine.reshape(-1, dir_enc_fine.shape[-1])
        )
        rgb_fine = rgb_fine.reshape(batch_size, num_fine, 3)
        density_fine = density_fine.reshape(batch_size, num_fine, 1)
        
        # Combine and render fine
        rgb_combined = torch.cat([rgb_coarse, rgb_fine], dim=1)
        density_combined = torch.cat([density_coarse, density_fine], dim=1)
        
        # Sort by depth
        sorted_indices = torch.argsort(depths_all, dim=-1)
        rgb_all = torch.zeros(batch_size, num_coarse + num_fine, 3, device=rgb_coarse.device)
        density_all = torch.zeros(batch_size, num_coarse + num_fine, 1, device=density_coarse.device)
        
        for i in range(batch_size):
            rgb_all[i] = rgb_combined[i, sorted_indices[i]]
            density_all[i] = density_combined[i, sorted_indices[i]]
        
        render_fine = volume_rendering(rgb_all, density_all, depths_all, rays_d, white_bg)
        
        return {
            'color_coarse': render_coarse['color'],
            'color_fine': render_fine['color'],
        }
    
    def train_step(self, rays_o, rays_d, target_rgb):
        """Single training step."""
        self.optimizer.zero_grad()
        
        # Render
        render_dict = self.render_rays(rays_o, rays_d)
        
        # Compute loss
        loss_coarse = self.criterion(render_dict['color_coarse'], target_rgb)
        loss_fine = self.criterion(render_dict['color_fine'], target_rgb)
        loss = loss_coarse + loss_fine
        
        # Backward
        loss.backward()
        self.optimizer.step()
        
        self.losses.append(loss.item())
        return loss.item()


# Demo: Create toy dataset and train briefly
print("Training Loop Demo")
print("=" * 50)

# Create small networks
pos_dim = pos_encoder.get_output_dim(3)
dir_dim = dir_encoder.get_output_dim(3)

coarse_net = NeRFNetwork(pos_dim, dir_dim, hidden_dim=128, num_layers=4)
fine_net = NeRFNetwork(pos_dim, dir_dim, hidden_dim=128, num_layers=4)

trainer = SimpleNeRFTrainer(coarse_net, fine_net, pos_encoder, dir_encoder, lr=1e-3)

# Toy training: random rays with synthetic target
print("\nTraining for 20 steps on random rays...")
num_train_steps = 20

for step in range(num_train_steps):
    # Random rays and targets
    batch_size = 32
    rays_o_train = torch.randn(batch_size, 3)
    rays_d_train = torch.randn(batch_size, 3)
    rays_d_train = rays_d_train / torch.norm(rays_d_train, dim=-1, keepdim=True)
    target_rgb_train = torch.rand(batch_size, 3)
    
    loss = trainer.train_step(rays_o_train, rays_d_train, target_rgb_train)
    
    if (step + 1) % 5 == 0:
        print(f"  Step {step + 1:3d}: Loss = {loss:.6f}")

# Plot loss curve
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(trainer.losses, linewidth=2, marker='o')
ax.set_xlabel('Training Step')
ax.set_ylabel('Loss')
ax.set_title('Training Loss Over Time')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\nFinal loss: {trainer.losses[-1]:.6f}")
print(f"Loss reduction: {trainer.losses[0] / trainer.losses[-1]:.2f}x")

## 8. Novel View Synthesis and Visualization

After training, we can render novel views from arbitrary camera poses:

1. **Generate new camera poses** (e.g., circular orbit around scene)
2. **Generate rays** from camera intrinsics/extrinsics
3. **Forward through trained networks**
4. **Render using volume rendering**
5. **Visualize depth maps and rendered images**

This demonstrates the core NeRF capability: **3D scene reconstruction without explicit geometry**.

In [ ]:
def generate_camera_pose(theta, radius=4.0):
    """Generate camera pose on circular trajectory."""
    cam_pos = torch.tensor([
        radius * np.cos(theta),
        0.0,
        radius * np.sin(theta),
    ], dtype=torch.float32)
    
    # Look at origin
    forward = -cam_pos
    forward = forward / torch.norm(forward)
    
    # Right vector
    up = torch.tensor([0.0, 1.0, 0.0])
    right = torch.cross(forward, up)
    right = right / torch.norm(right)
    
    # Recompute up
    up = torch.cross(right, forward)
    up = up / torch.norm(up)
    
    # Rotation matrix
    R = torch.stack([right, up, -forward], dim=0)
    
    # Pose matrix
    pose = torch.eye(4)
    pose[:3, :3] = R
    pose[:3, 3] = cam_pos
    
    return pose


# Render novel views
print("Novel View Synthesis")
print("=" * 50)

height, width = 64, 64
focal = 50.0

# Generate views on circular trajectory
num_views = 4
poses_novel = []
for i in range(num_views):
    theta = 2 * np.pi * i / num_views
    pose = generate_camera_pose(theta, radius=4.0)
    poses_novel.append(pose)

# Render views
fig, axes = plt.subplots(2, num_views, figsize=(15, 6))

print(f"\nRendering {num_views} novel views...")

with torch.no_grad():
    for view_idx, pose in enumerate(poses_novel):
        # Generate rays
        rays_o, rays_d = get_rays(height, width, focal, pose)
        
        # Render
        render_dict = trainer.render_rays(
            rays_o.reshape(-1, 3),
            rays_d.reshape(-1, 3),
            num_coarse=16, num_fine=32
        )
        
        # Reshape
        color_coarse = render_dict['color_coarse'].reshape(height, width, 3)
        color_fine = render_dict['color_fine'].reshape(height, width, 3)
        
        # Visualize coarse
        axes[0, view_idx].imshow(torch.clamp(color_coarse, 0, 1).numpy())
        axes[0, view_idx].set_title(f'View {view_idx + 1} (Coarse)')
        axes[0, view_idx].axis('off')
        
        # Visualize fine
        axes[1, view_idx].imshow(torch.clamp(color_fine, 0, 1).numpy())
        axes[1, view_idx].set_title(f'View {view_idx + 1} (Fine)')
        axes[1, view_idx].axis('off')

plt.suptitle('Novel View Synthesis Results', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print("\nNovel view synthesis complete!")
print("The rendered views show the learned radiance field from new viewpoints.")
print("\nKey insights:")
print("  - Coarse network: Fast estimate with stratified sampling")
print("  - Fine network: Refined estimate with importance sampling")
print("  - Both are fully differentiable and trained end-to-end")
print("  - No explicit 3D geometry needed!")

## Summary: NeRF as Neural Ray Tracing

**NeRF replaces classical ray tracing with a learned implicit representation:**

| Classical Ray Tracing | NeRF (Neural Ray Tracing) |
|---|---|
| **Explicit geometry**: Meshes, SDFs | **Implicit representation**: MLP |
| **Material properties**: BRDF, textures | **Learned appearance**: View-dependent RGB |
| **Hard surface intersections**: Ray-surface hits | **Soft volumetric accumulation**: Alpha compositing |
| **Recursive bounces**: Multiple rays per pixel | **Direct path integration**: Volume rendering |
| **Manual optimization**: Parameter tuning | **Automatic optimization**: Gradient descent |

**Mathematical Pipeline**:

$$\text{Render}(\mathbf{r}) = \int_{t_n}^{t_f} T(t) \sigma(\gamma(\mathbf{r}(t))) \cdot \text{RGB}(\gamma(\mathbf{r}(t)), \gamma(\mathbf{d})) \, dt$$

Where:
- $\gamma(\cdot)$ is **positional encoding** (Fourier features)
- $\sigma(\cdot)$ is the **learned density field** (MLP output)
- $\text{RGB}(\cdot, \cdot)$ is the **learned color field** (MLP output)
- $T(t)$ is **transmittance** (accumulated opacity)

**Key Advantages**:
✓ **Compact representation**: Few parameters for entire scene  
✓ **Photorealistic**: Captures view-dependent effects  
✓ **Differentiable**: End-to-end training from 2D images  
✓ **Fast rendering**: No explicit ray-surface computation  
✓ **Flexible**: Extends to video, dynamic scenes, etc.

This notebook demonstrates the complete NeRF pipeline from first principles!

In [ ]:
def get_rays(height, width, focal, pose):
    """
    Generate rays from camera intrinsics and extrinsics.
    
    Args:
        height, width: Image dimensions
        focal: Focal length
        pose: 4x4 extrinsic matrix [R | t]
    
    Returns:
        rays_o: Ray origins [height, width, 3]
        rays_d: Ray directions [height, width, 3] (normalized)
    """
    # Create pixel coordinates
    x = torch.arange(width, dtype=torch.float32, device=device)
    y = torch.arange(height, dtype=torch.float32, device=device)
    xx, yy = torch.meshgrid(x, y, indexing='ij')
    
    # Normalize to camera space
    coords = torch.stack([
        (xx - width / 2.0) / focal,
        -(yy - height / 2.0) / focal,  # Negative: y pixel increases downward
        torch.ones_like(xx),
    ], dim=-1)  # [width, height, 3]
    
    # Transform from camera to world space
    pose = pose.to(device)
    R = pose[:3, :3]
    t = pose[:3, 3]
    
    # Inverse transform (since pose is camera-to-world)
    R_inv = R.T
    
    # Transform directions
    dirs = torch.matmul(coords, R_inv.T)
    dirs = dirs / (torch.norm(dirs, dim=-1, keepdim=True) + 1e-8)
    
    # Ray origins (camera center in world space)
    rays_o = -torch.matmul(R_inv, t)
    rays_o = rays_o.expand(*coords.shape[:-1], 3)
    
    # Swap to [height, width, 3]
    return rays_o.transpose(0, 1), dirs.transpose(0, 1)


# Test ray generation
print("Ray Generation Test")
print("=" * 50)

height, width = 64, 64
focal = 50.0

# Simple identity pose (camera at origin, looking down +z)
pose = torch.eye(4)
pose[1, 3] = 0.0  # Slight vertical offset

rays_o, rays_d = get_rays(height, width, focal, pose)

print(f"Image size: {height} x {width}")
print(f"Focal length: {focal:.2f}")
print(f"Ray origins shape: {rays_o.shape}")
print(f"Ray directions shape: {rays_d.shape}")

# Visualize rays from camera center
fig = plt.figure(figsize=(14, 6))

# 2D projection
ax1 = fig.add_subplot(121)
ax1.scatter(rays_o[0, :, 0].cpu(), rays_o[0, :, 2].cpu(), s=1, alpha=0.3)
ax1.set_xlabel('X')
ax1.set_ylabel('Z')
ax1.set_title('Ray Origins (Top View)')
ax1.grid(True, alpha=0.3)
ax1.axis('equal')

# 3D visualization
ax2 = fig.add_subplot(122, projection='3d')
indices = [10, 32, 54]
for i in indices:
    for j in indices:
        o = rays_o[i, j].cpu().numpy()
        d = rays_d[i, j].cpu().numpy()
        ax2.quiver(o[0], o[1], o[2], d[0]*0.5, d[1]*0.5, d[2]*0.5, 
                  length=1.0, normalize=False, alpha=0.6)

ax2.set_xlabel('X')
ax2.set_ylabel('Y')
ax2.set_zlabel('Z')
ax2.set_title('Ray Directions (Selected Pixels)')
ax2.set_xlim([-2, 2])
ax2.set_ylim([-2, 2])
ax2.set_zlim([-3, 1])

plt.tight_layout()
plt.show()

# Check properties
print(f"\nRay properties:")
print(f"  Origins range: X [{rays_o[..., 0].min():.2f}, {rays_o[..., 0].max():.2f}]")
print(f"  Directions normalized: {torch.norm(rays_d, dim=-1).unique().item():.6f}")
print(f"  All directions point in similar direction: {rays_d[0, 0] - rays_d[height//2, width//2]}")

## 5. Stratified and Hierarchical Sampling

**Stratified Sampling**: Divide ray segment $[t_{near}, t_{far}]$ into uniform bins, then sample randomly within each bin.

$$t_i = t_{near} + \frac{i}{N} (t_{far} - t_{near}) + \epsilon$$

where $N$ is number of samples, $\epsilon \sim \mathcal{U}(0, \Delta t)$ is random perturbation.

**Hierarchical Importance Sampling**: Use coarse network density estimates to weight sampling distribution for fine network, concentrating samples where the ray has high opacity.

This two-stage approach is crucial for efficiency—coarse network identifies important regions, fine network refines rendering there.

In [ ]:
def stratified_sample(rays_o, rays_d, near, far, num_samples, perturb=True):
    """
    Stratified sampling along rays.
    
    Args:
        rays_o: Ray origins [batch, 3]
        rays_d: Ray directions [batch, 3]
        near, far: Depth range
        num_samples: Number of samples per ray
        perturb: Add random perturbation
    
    Returns:
        sample_points: 3D coordinates [batch, num_samples, 3]
        depths: Depth values [batch, num_samples]
    """
    batch_size = rays_o.shape[0]
    
    # Linearly spaced depths
    depths = torch.linspace(near, far, num_samples, device=device)
    depths = depths.unsqueeze(0).expand(batch_size, -1)  # [batch, num_samples]
    
    if perturb:
        # Add random perturbation within each bin
        bin_width = (far - near) / num_samples
        depths = depths + (torch.rand_like(depths) - 0.5) * bin_width
        depths = torch.clamp(depths, near, far)
    
    # Compute 3D points: P = O + d * D
    sample_points = rays_o.unsqueeze(1) + rays_d.unsqueeze(1) * depths.unsqueeze(-1)
    
    return sample_points, depths


def hierarchical_sample(rays_o, rays_d, depths_coarse, weights, num_samples, perturb=True):
    """
    Importance sampling based on coarse density weights.
    
    Args:
        depths_coarse: Coarse samples [batch, num_coarse]
        weights: Density-based weights [batch, num_coarse]
        num_samples: Number of fine samples
    
    Returns:
        sample_points: Fine samples [batch, num_samples, 3]
        depths_new: Fine depths [batch, num_samples]
        depths_all: Combined and sorted depths [batch, num_coarse + num_samples]
    """
    batch_size = rays_o.shape[0]
    
    # Compute bin edges from coarse depths
    depths_sorted, _ = torch.sort(depths_coarse, dim=-1)
    
    bin_edges = torch.cat([
        depths_sorted[..., :1],
        (depths_sorted[..., :-1] + depths_sorted[..., 1:]) / 2.0,
        depths_sorted[..., -1:],
    ], dim=-1)
    
    # Convert weights to probability distribution
    pdf = weights / (torch.sum(weights, dim=-1, keepdim=True) + 1e-5)
    cdf = torch.cumsum(pdf, dim=-1)
    cdf = torch.cat([torch.zeros_like(cdf[..., :1]), cdf], dim=-1)
    
    # Inverse transform sampling
    uniform_samples = torch.rand(batch_size, num_samples, device=device)
    indices = torch.searchsorted(cdf, uniform_samples, right=False) - 1
    indices = torch.clamp(indices, 0, cdf.shape[-1] - 2)
    
    left_edges = bin_edges[torch.arange(batch_size).unsqueeze(-1), indices]
    right_edges = bin_edges[torch.arange(batch_size).unsqueeze(-1), indices + 1]
    
    left_cdf = cdf[torch.arange(batch_size).unsqueeze(-1), indices]
    right_cdf = cdf[torch.arange(batch_size).unsqueeze(-1), indices + 1]
    
    # Linear interpolation
    cdf_width = torch.clamp(right_cdf - left_cdf, min=1e-5)
    t = (uniform_samples - left_cdf) / cdf_width
    t = torch.clamp(t, 0, 1)
    
    depths_new = left_edges + t * (right_edges - left_edges)
    
    # Compute fine sample points
    sample_points = rays_o.unsqueeze(1) + rays_d.unsqueeze(1) * depths_new.unsqueeze(-1)
    
    # Combine and sort all depths
    depths_all = torch.cat([depths_coarse, depths_new], dim=-1)
    depths_all, _ = torch.sort(depths_all, dim=-1)
    
    return sample_points, depths_new, depths_all


# Demonstrate sampling
print("Sampling Demonstration")
print("=" * 50)

batch_size = 4
near, far = 2.0, 6.0
num_samples_coarse = 16

# Use rays from previous section
sample_indices = torch.tensor([0, 10, 32, 54])
rays_o_batch = rays_o.reshape(-1, 3)[torch.arange(height*width).reshape(height, width)[sample_indices, sample_indices]]
rays_d_batch = rays_d.reshape(-1, 3)[torch.arange(height*width).reshape(height, width)[sample_indices, sample_indices]]

# Stratified sampling
sample_points_coarse, depths_coarse = stratified_sample(
    rays_o_batch, rays_d_batch, near, far, num_samples_coarse, perturb=True
)

print(f"Stratified sampling:")
print(f"  Depths range: [{depths_coarse.min():.3f}, {depths_coarse.max():.3f}]")
print(f"  Sample points shape: {sample_points_coarse.shape}")

# Simulate coarse network density for hierarchical sampling
density_coarse = torch.exp(-((depths_coarse - 4.0) ** 2) / 0.5)  # Gaussian at depth 4
weights_coarse = density_coarse * 0.1  # Simulate weights

num_samples_fine = 12
sample_points_fine, depths_fine, depths_all = hierarchical_sample(
    rays_o_batch, rays_d_batch, depths_coarse, weights_coarse, num_samples_fine
)

print(f"\nHierarchical sampling:")
print(f"  Fine depths shape: {depths_fine.shape}")
print(f"  Combined depths shape: {depths_all.shape}")

# Visualize sampling
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Coarse sampling
ax = axes[0]
ax.hist(depths_coarse.cpu().numpy().flatten(), bins=10, alpha=0.5, label='Coarse')
ax.set_xlabel('Depth')
ax.set_ylabel('Count')
ax.set_title('Stratified Sampling')
ax.grid(True, alpha=0.3)
ax.legend()

# Hierarchical sampling
ax = axes[1]
ax.hist(depths_coarse.cpu().numpy().flatten(), bins=10, alpha=0.5, label='Coarse')
ax.hist(depths_fine.cpu().numpy().flatten(), bins=10, alpha=0.5, label='Fine')
ax.plot(depths_coarse.cpu().numpy()[0], weights_coarse.cpu().numpy()[0] * 100, 'r-', label='Weights')
ax.set_xlabel('Depth')
ax.set_ylabel('Count')
ax.set_title('Hierarchical Importance Sampling')
ax.grid(True, alpha=0.3)
ax.legend()

plt.tight_layout()
plt.show()

## 6. Volumetric Rendering with Alpha Compositing

**Volume Rendering Equation**:

$$C(\mathbf{r}) = \int_{t_n}^{t_f} T(t) \sigma(\mathbf{r}(t)) \mathbf{c}(\mathbf{r}(t), \mathbf{d}) \, dt$$

where:
- $T(t) = \exp\left(-\int_{t_n}^t \sigma(\mathbf{r}(s)) ds\right)$ is transmittance
- $\sigma$ is volume density at 3D point
- $\mathbf{c}$ is color at that point

**Discrete Alpha Compositing** (Riemann approximation):

$$T_i = \exp\left(-\sum_{j=1}^{i-1} \sigma_j \delta_j\right)$$
$$\alpha_i = 1 - \exp(-\sigma_i \delta_i)$$
$$C = \sum_i T_i \alpha_i \mathbf{c}_i + T_N \mathbf{c}_{bg}$$

This is **fully differentiable** with respect to network outputs!